# 03 · Join Sofascore + Capology — Spain La Liga 21/22

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2021/22 de La Liga española**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_spain_2122.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_spain_2122.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  604 jugadores | 116 columnas
Capology:   572 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   deportivo alaves
   levante ud

En Capology pero no en Sofascore:
   alaves
   levante


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'alaves':'deportivo alaves',
            'levante':'levante ud'

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 508/604 (84.1%)
Sin emparejar: 96


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          9
Revisión media    (0.75 ≤ score < 0.90):   12
Revisión estricta (0.50 ≤ score < 0.75):   44
Revisión muy est. (score < 0.50):           31


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
41,Alexander Sørloth,Real Sociedad,alexander sorloth,0.970
43,Rober Ibáñez,Osasuna,robert ibanez,0.960
19,Jens Jønsson,Cádiz,jens jonsson,0.957
84,Javier Ontiveros,Osasuna,javi ontiveros,0.933
38,Manuel Sánchez,Osasuna,manu sanchez,0.923
9,Daniel Vivian,Athletic Club,dani vivian,0.917
14,Yéremy Pino,Villarreal,yeremi pino,0.909
75,Javier Díaz,Sevilla,javi diaz,0.900
45,Dani Raba,Granada,daniel raba,0.900


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
55,Nicolás Melamed,Espanyol,nico melamed,0.889
18,Matthew Miazga,Deportivo Alavés,matt miazga,0.880
10,Abdessamad Ezzalzouli,Barcelona,abde ezzalzouli,0.833
44,José Juan Macías,Getafe,jose macias,0.815
28,Florentino Luís,Getafe,florentino,0.800
5,José María Giménez,Atlético Madrid,jose gimenez,0.800
53,Alejandro Asensio,Rayo Vallecano,alejandro catena,0.788
25,José Luis Gayà,Valencia,jose gaya,0.783
62,Álex Petxarroman,Athletic Club,alex petxa,0.769
39,Fernando Niño,Mallorca,fer nino,0.762


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['alejandro asensio',
                      'alejandro primo'
]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')

Aceptados: 10 | Excluidos: 2


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
26,Pathé Ismaël Ciss,Rayo Vallecano,pathe ciss,0.741
21,Maximiliano Gómez,Valencia,maxi gomez,0.741
70,Ivan Barbero,Osasuna,barbero,0.737
22,Anthony Lozano,Cádiz,choco lozano,0.692
82,Cristhian Mosquera,Valencia,cristian rivero,0.667
68,Álvaro Aguirre,Rayo Vallecano,alvaro garcia,0.667
92,Hugo Sotelo,Celta Vigo,hugo mallo,0.667
78,Mario Domínguez,Valencia,maxi gomez,0.640
57,Ander Martín,Real Sociedad,ander guevara,0.640
31,Hugo Álvarez,Celta Vigo,hugo mallo,0.636


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['pathe ismael ciss',
                    'maximiliano gomez',
                    'ivan barbero',
                    'anthony lozano',
                    'radamel falcao',
                    'rafinha alcantara',
                    'pablo gavi',
                    'wu lei'
]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 8


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
48,Alexandru Țîrlea,Deportivo Alavés,mamadou sylla,0.483
64,Estanis Pedrola,Barcelona,inaki pena,0.480
83,Pedro Benito,Cádiz,ruben sobrino,0.480
74,Juanlu Sánchez,Sevilla,joan jordan,0.480
54,Peter González,Real Madrid,gareth bale,0.480
76,Cristo Romero,Real Sociedad,mikel merino,0.480
61,Gabri Veiga,Celta Vigo,javi galan,0.476
73,Iker Benito,Osasuna,kike barja,0.476
77,Álvaro Sanz Catalán,Barcelona,alejandro balde,0.471
90,Raúl García de Haro,Real Betis,andres guardado,0.471


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 535/604 (88.6%)
Sin salario:     69


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 69


,player,team,minutesPlayed,appearances,goals,assists
0,Saúl Ñíguez,Atlético Madrid,200,3,0,1
1,Javi Serrano,Atlético Madrid,69,5,0,0
2,Carlos Martín,Atlético Madrid,8,1,0,0
3,Giuliano Simeone,Atlético Madrid,1,1,0,0
4,Ilias Akhomach,Barcelona,125,2,0,0
5,Emerson Royal,Barcelona,90,3,0,0
6,Álvaro Sanz Catalán,Barcelona,27,2,0,0
7,Estanis Pedrola,Barcelona,10,1,0,0
8,Mika Mármol,Barcelona,1,1,0,0
9,Gabri Veiga,Celta Vigo,126,7,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Atlético Madrid  —  SF sin salario:


,player,minutesPlayed
0,Carlos Martín,8
1,Giuliano Simeone,1
2,Javi Serrano,69
3,Saúl Ñíguez,200


  CG plantilla completa:


,player,player_norm
0,Ángel Correa,angel correa
1,Antoine Griezmann,antoine griezmann
2,Benjamin Lecomte,benjamin lecomte
3,Daniel Wass,daniel wass
4,Felipe,felipe
5,Francisco Montero,francisco montero
6,Geoffrey Kondogbia,geoffrey kondogbia
7,Héctor Herrera,hector herrera
8,Ivan Saponjic,ivan saponjic
9,Jan Oblak,jan oblak



  Barcelona  —  SF sin salario:


...

,player,player_norm
0,Álex Blanco,alex blanco
1,Bryan Gil,bryan gil
2,Carlos Soler,carlos soler
3,Cristian Rivero,cristian rivero
4,Cristiano Piccini,cristiano piccini
5,Daniel Wass,daniel wass
6,Denis Cheryshev,denis cheryshev
7,Dimitri Foulquier,dimitri foulquier
8,Eray Cömert,eray comert
9,Gabriel Paulista,gabriel paulista



  Villarreal  —  SF sin salario:


,player,minutesPlayed
0,Carlo Adriano García,9
1,Nicolas Jackson,162


  CG plantilla completa:


,player,player_norm
0,Aïssa Mandi,aissa mandi
1,Alberto Moreno,alberto moreno
2,Alfonso Pedraza,alfonso pedraza
3,Arnaut Danjuma,arnaut danjuma
4,Boulaye Dia,boulaye dia
5,Dani Parejo,dani parejo
6,Daniel Raba,daniel raba
7,Étienne Capoue,etienne capoue
8,Francis Coquelin,francis coquelin
9,Gerard Moreno,gerard moreno


In [23]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('rodri sanchez', 'real betis'): ('rodri', 'real betis'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 1


In [24]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: rodri sanchez (real betis) → rodri (real betis)

Tras matches manuales: 536/604 (88.7%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [25]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_spain_2122.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_spain_2122.csv
   Jugadores totales:  604
   Con salario:        536
   Sin salario (NaN):  68
   Columnas:           121
